# 02_exploratory_analysis.ipynb

Explore risk patterns and create the analyst-facing charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Python":
    PROJECT_ROOT = PROJECT_ROOT.parent
sns.set_theme(style="whitegrid")
NAVY, RED = "#1F3B63", "#C0392B"
df = pd.read_csv(PROJECT_ROOT / "Data" / "Cleaned" / "credit_risk_clean.csv")
print("Data shape:", df.shape)
print(df.head())

## Default rate and interest rate by grade

In [ ]:
g = df.groupby("loan_grade").agg(default_rate=("loan_status", "mean"), avg_rate=("int_rate_filled", "median")).reset_index()
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(g.loan_grade, g.default_rate * 100, color=NAVY, label="Default rate")
ax1.set_ylabel("Default rate (%)")
ax2 = ax1.twinx()
ax2.plot(g.loan_grade, g.avg_rate, color=RED, marker="o", linewidth=2, label="Median interest rate")
ax2.set_ylabel("Median interest rate (%)")
ax1.set_title("Default rate and interest rate by grade")
fig.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "Images" / "py-grade-vs-rate.png", dpi=150)
plt.show()

## Loan-to-income bands

In [ ]:
bands = df.groupby("loan_to_income_band").agg(count=("loan_status", "size"), default_rate=("loan_status", "mean")).reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=bands, x="loan_to_income_band", y="default_rate", color=NAVY, ax=ax)
ax.set_title("Default rate by loan-to-income band")
ax.set_ylabel("Default rate")
plt.tight_layout()
plt.show()

## Loan purpose

In [ ]:
purpose = df.groupby("loan_intent")["loan_status"].agg(["count", "mean"]).sort_values("mean", ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=purpose.reset_index(), x="loan_intent", y="mean", color=NAVY, ax=ax)
ax.set_title("Default rate by loan purpose")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## Home ownership

In [ ]:
ownership = df.groupby("person_home_ownership")["loan_status"].agg(["count", "mean"]).reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=ownership, x="person_home_ownership", y="mean", color=RED, ax=ax)
ax.set_title("Default rate by home ownership")
plt.tight_layout()
plt.show()

## Prior default

In [ ]:
prior = df.groupby("cb_person_default_on_file")["loan_status"].agg(["count", "mean"]).reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=prior, x="cb_person_default_on_file", y="mean", color=NAVY, ax=ax)
ax.set_title("Default rate by prior default history")
plt.tight_layout()
plt.show()

## Age

In [ ]:
age_groups = pd.cut(df["person_age"], bins=[18,24,29,34,39,44,49,100], labels=["19-24","25-29","30-34","35-39","40-44","45-49","50-100"])
summary = pd.DataFrame({"age_group": age_groups, "default": df["loan_status"]}).groupby("age_group")["default"].agg(["count", "mean"])
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=summary.reset_index(), x="age_group", y="mean", color=RED, ax=ax)
ax.set_title("Default rate by age group")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Income

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="loan_status", y="person_income", ax=ax, color=NAVY)
ax.set_title("Income distribution by loan status")
plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
numeric = df.select_dtypes(include=[np.number]).corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(numeric, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation heatmap of numeric columns")
plt.tight_layout()
plt.show()

## KDE plot of loan-to-income

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="loan_percent_income", hue="loan_status", fill=True, common_norm=False, palette=[NAVY, RED])
plt.title("Defaulted loans cluster at higher loan-to-income ratios")
plt.xlabel("Loan-to-income ratio")
plt.ylabel("Density")
plt.tight_layout()
plt.show()